# Working with MERFISH data

This tutorial reads MERFISH data into stLearn and runs a basic clustering workflow.

The data are from [Xia et al. (2019), *PNAS* 116(39):19490](https://doi.org/10.1073/pnas.1912459116): MERFISH profiling of cultured U-2 OS cells. We use two of the paper's supplementary files:

- `pnas.1912459116.sd12.csv`: the count matrix (genes × cells)
- `pnas.1912459116.sd15.xlsx`: the cell positions in microns

The original PNAS download links no longer work, so we download the same files from the copy kept in the Giotto team's [spatial-datasets](https://github.com/drieslab/spatial-datasets) repository on GitHub.

In [ ]:
import stlearn as st

# Setting figure parameters
st.settings.set_figure_params(dpi=120)

Read MERFISH data follow the instruction from scanpy

In [ ]:
import urllib.request
from pathlib import Path

base_url = (
    "https://raw.githubusercontent.com/drieslab/spatial-datasets/master/"
    "data/2019_merFISH_PNAS/raw_data/"
)
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

count_file = data_dir / "pnas.1912459116.sd12.csv"
spatial_file = data_dir / "pnas.1912459116.sd15.xlsx"

for path in (count_file, spatial_file):
    if not path.exists():
        urllib.request.urlretrieve(base_url + path.name, path)

## Read the data

`st.io.read_merfish` keeps the cells that have a position in the spreadsheet, stores the positions in `obsm['spatial']` and creates a blank background image, so the stLearn plotting functions work without a tissue image.

In [ ]:
data = st.io.read_merfish(
    count_matrix_file=count_file,
    spatial_file=spatial_file,
)
data

## Quality check

Each dot is a cell, sized by its total counts and coloured by the number of genes detected.

In [ ]:
st.pl.qc_plot(data, name="merfish_qc")

## Preprocessing

In [ ]:
st.pp.filter_genes(data,min_cells=3)
st.pp.normalize_total(data)
st.pp.log1p(data)
st.pp.scale(data)

Running PCA to reduce the dimensions to 50

In [ ]:
st.em.run_pca(data,n_comps=50,random_state=0)

## Leiden clustering

In [ ]:
st.pp.neighbors(data,n_neighbors=25)

In [ ]:
st.tl.clustering.leiden(data)

In [ ]:
st.pl.cluster_plot(data,use_label="leiden",size=10)